## CREATE silver_source 

In [0]:
spark.sql("""
        SELECT *, 
            upper(customer_name) AS customer_name_upper,
            date(current_timestamp()) AS process_timestamp
        FROM datamodeling.bronze.bronze_table
        """).createOrReplaceTempView("silver_source")       # silver_source taking from the bronze_table from the previous phase

In [0]:
%sql
SELECT * FROM silver_source

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,customer_name_upper,process_timestamp
1004,2024-07-02,4,David Kim,david@hotmail.com,504,Samsung Galaxy S23,Electronics,1,799.99,Apple Pay,USA,2024-07-02,DAVID KIM,2026-07-12
1005,2024-07-02,5,Eve Lee,eve@gmail.com,505,Apple Watch,Electronics,1,399.99,Google Pay,USA,2024-07-02,EVE LEE,2026-07-12


## MERGE USING PySpark

In [0]:
# One is source, and one is destination (target)
if spark.catalog.tableExists('datamodeling.silver.silver_table'):                # check if destination table already exists
    pass

else:                # if not, then create
    spark.sql("""
              CREATE TABLE IF NOT EXISTS datamodeling.silver.silver_table 
              AS 
              SELECT * FROM silver_source
            """)

## MERGE USING SQL

In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.silver.silver_table
AS
SELECT * FROM silver_source

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- MERGE COMMAND (Incremental Load)
MERGE INTO datamodeling.silver.silver_table AS target
USING silver_source AS source
ON target.order_id = source.order_id

-- UPSERT
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
SELECT * FROM datamodeling.silver.silver_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated,customer_name_upper,process_timestamp
1004,2024-07-02,4,David Kim,david@hotmail.com,504,Samsung Galaxy S23,Electronics,1,799.99,Apple Pay,USA,2024-07-02,DAVID KIM,2026-07-12
1005,2024-07-02,5,Eve Lee,eve@gmail.com,505,Apple Watch,Electronics,1,399.99,Google Pay,USA,2024-07-02,EVE LEE,2026-07-12
1001,2024-07-01,1,Alice Johnson,alice@gmail.com,501,iPhone 14,Electronics,1,999.99,Credit Card,USA,2024-07-01,ALICE JOHNSON,2026-07-12
1002,2024-07-01,2,Bob Smith,bob@yahoo.com,502,AirPods Pro,Electronics,2,199.99,Paypal,USA,2024-07-01,BOB SMITH,2026-07-12
1003,2024-07-01,3,Charlie Brown,charlie@outlook.com,503,Nike Shoes,Footwear,1,129.99,Credit Card,Canada,2024-07-01,CHARLIE BROWN,2026-07-12
